In [5]:
import torch
from monai.data import Dataset, DataLoader 
from monai.networks.nets import UNet, AttentionUnet, UNETR
from monai.transforms import Compose, LoadImaged, ScaleIntensityd, RandFlipd, ToTensord, Activations, AsDiscrete, EnsureChannelFirstd
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
import pandas as pd
import os
from pathlib import Path

In [6]:
CWD = Path(os.getcwd()).parent
print(CWD)

def prepare_splits(labels_csv_path):
    labels_df = pd.read_csv(labels_csv_path)
    train_df = labels_df[labels_df['split'] == 'train']
    val_df = labels_df[labels_df['split'] == 'val']
    test_df = labels_df[labels_df['split'] == 'test']
    return train_df, val_df, test_df

def get_dataset_paths(dataset_name):
    CWD = Path(os.getcwd()).parent
    valid_datasets = ['NuInSeg', 'PanNuke', 'MoNuSeg']
    if dataset_name not in valid_datasets:
        raise ValueError(f"Invalid dataset name: {dataset_name}. Valid options are: {valid_datasets}")
    image_path = Path(CWD / f'data/processed/{dataset_name}/Images/')
    mask_path = Path(CWD / f'data/processed/{dataset_name}/Masks/')
    return image_path, mask_path

def prepare_datasets(df, dataset_name):
    image_path, mask_path = get_dataset_paths(dataset_name)
    print(image_path)
    images = [str(image_path / f"{row['FileName']}.npy") for _, row in df.iterrows()]
    masks = [str(mask_path / f"{row['FileName']}.npy") for _, row in df.iterrows()]
    monai_dict = [{'image': image, 'label': mask} for image, mask in zip(images, masks)]
    return monai_dict

/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos


# Definindo as redes utilizadas

# Configurando os Datasets

In [7]:
MoNuSeg_csv = CWD / 'data/processed/MoNuSeg/labels.csv'
PanNuke_csv = CWD / 'data/processed/PanNuke/labels.csv'
NuInSeg_csv = CWD / 'data/processed/NuInSeg/labels.csv'

MoNuSeg_train_df, MoNuSeg_val_df, MoNuSeg_test_df = prepare_splits(MoNuSeg_csv)
PanNuke_train_df, PanNuke_val_df, PanNuke_test_df = prepare_splits(PanNuke_csv)
NuInSeg_train_df, NuInSeg_val_df, NuInSeg_test_df = prepare_splits(NuInSeg_csv)

MoNuSeg_train_data = prepare_datasets(MoNuSeg_train_df, 'MoNuSeg')
PanNuke_train_data = prepare_datasets(PanNuke_train_df, 'PanNuke')
NuInSeg_train_data = prepare_datasets(NuInSeg_train_df, 'NuInSeg')

MoNuSeg_val_data = prepare_datasets(MoNuSeg_val_df, 'MoNuSeg')
PanNuke_val_data = prepare_datasets(PanNuke_val_df, 'PanNuke')
NuInSeg_val_data = prepare_datasets(NuInSeg_val_df, 'NuInSeg')


transforms_train = Compose([
    LoadImaged(keys=['image', 'label'], reader="NumpyReader"),

    EnsureChannelFirstd(keys=['image'], channel_dim=-1),
    EnsureChannelFirstd(keys=['label'], channel_dim="no_channel"),

    ScaleIntensityd(keys=['image']),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
    ToTensord(keys=['image', 'label'])
    ])

transforms_val = Compose([
    LoadImaged(keys=['image', 'label'], reader="NumpyReader"),
    EnsureChannelFirstd(keys=['image'], channel_dim=-1),
    EnsureChannelFirstd(keys=['label'], channel_dim="no_channel"),
    ScaleIntensityd(keys=['image']),
    ToTensord(keys=['image', 'label'])
    ])


MoNuSeg_train_ds = Dataset(data=MoNuSeg_train_data, transform=transforms_train)
PanNuke_train_ds = Dataset(data=PanNuke_train_data, transform=transforms_train)
NuInSeg_train_ds = Dataset(data=NuInSeg_train_data, transform=transforms_train)

MoNuSeg_val_ds = Dataset(data=MoNuSeg_val_data, transform=transforms_val)
PanNuke_val_ds = Dataset(data=PanNuke_val_data, transform=transforms_val)
NuInSeg_val_ds = Dataset(data=NuInSeg_val_data, transform=transforms_val)

datasets = {
    'MoNuSeg': {
        'train': MoNuSeg_train_ds,
        'val': MoNuSeg_val_ds
    },
    'PanNuke': {
        'train': PanNuke_train_ds,
        'val': PanNuke_val_ds
    },
    'NuInSeg': {    
        'train': NuInSeg_train_ds,
        'val': NuInSeg_val_ds
        }
}        


/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/MoNuSeg/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/PanNuke/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/NuInSeg/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/MoNuSeg/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/PanNuke/Images
/home/vinicius/IA901/IA901-2026S1/projetos/segmentacao_de_nucleos/data/processed/NuInSeg/Images


# Treinando UNET

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for dataset_name, dataset_splits in datasets.items():
    print(f"\nIniciando treinamento para o dataset: {dataset_name}")
    
    train_loader = DataLoader(dataset_splits['train'], batch_size=16, shuffle=True)
    val_loader = DataLoader(dataset_splits['val'], batch_size=16, shuffle=False)

    unet = UNet( # unet do monai
        spatial_dims=2,
        in_channels=3,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
    ).to(device)

    model = unet

    loss_function = DiceCELoss(sigmoid=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    dice_metric = DiceMetric(include_background=True, reduction="mean")

    post_pred = Compose([Activations(keys="pred", sigmoid=True), AsDiscrete(keys="pred", threshold=0.5)])
    post_label = Compose([AsDiscrete(threshold=0.5)])

    max_epochs = 50
    val_interval = 1 
    best_metric = -1
    best_metric_epoch = -1

    for epoch in range(max_epochs):
        print("-" * 10)
        print(f"Época {epoch + 1}/{max_epochs}")
        
        # --- FASE DE TREINO ---
        model.train()
        epoch_loss = 0
        step = 0
        
        for batch_data in train_loader:
            step += 1
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # Calcula a Loss combinada (Dice + Cross Entropy)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        epoch_loss /= step
        print(f"Loss média de Treino: {epoch_loss:.4f}")

        # --- FASE DE VALIDAÇÃO ---
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                for val_data in val_loader:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    
                    # Predição do modelo
                    val_outputs = model(val_inputs)
                    
                    # Aplica o pós-processamento (limiar de 0.5)
                    val_outputs = [post_pred(i) for i in val_outputs]
                    val_labels = [post_label(i) for i in val_labels]
                    
                    # Computa a métrica Dice para o batch atual
                    dice_metric(y_pred=val_outputs, y=val_labels)
                
                # Agrega o resultado de toda a validação
                metric = dice_metric.aggregate().item()
                dice_metric.reset() # Reseta para a próxima validação
                
                print(f"Métrica Dice na Validação: {metric:.4f}")
                
                # Salva o melhor modelo baseado no Score Dice
                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    torch.save(model.state_dict(), f"results/models/{model.__class__.__name__.lower()}/{dataset_name}_{model.__class__.__name__}.pth")
                    print("Encontrado um novo melhor modelo! Salvo com sucesso.")
    print(f"\nTreino concluído! Melhor Dice: {best_metric:.4f} na época {best_metric_epoch}")
                    




Iniciando treinamento para o dataset: MoNuSeg
----------
Época 1/50
Loss média de Treino: 1.4208
Métrica Dice na Validação: 0.3858
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 2/50
Loss média de Treino: 1.3349
Métrica Dice na Validação: 0.4707
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 3/50
Loss média de Treino: 1.2746
Métrica Dice na Validação: 0.5594
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 4/50
Loss média de Treino: 1.2324
Métrica Dice na Validação: 0.6029
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 5/50
Loss média de Treino: 1.2017
Métrica Dice na Validação: 0.6431
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 6/50
Loss média de Treino: 1.1834
Métrica Dice na Validação: 0.6714
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 7/50
Loss média de Treino: 1.1729
Métrica Dice na Validação: 0.6931
Encontrado um novo melhor modelo! Salvo 

# Treinando AttentionUnet

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for dataset_name, dataset_splits in datasets.items():
    print(f"\nIniciando treinamento para o dataset: {dataset_name}")
    
    train_loader = DataLoader(dataset_splits['train'], batch_size=16, shuffle=True)
    val_loader = DataLoader(dataset_splits['val'], batch_size=16, shuffle=False)

    attention_unet = AttentionUnet( # attention unet do monai
        spatial_dims=2,
        in_channels=3,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2)
    ).to(device)

    model = attention_unet

    loss_function = DiceCELoss(sigmoid=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    dice_metric = DiceMetric(include_background=True, reduction="mean")

    post_pred = Compose([Activations(keys="pred", sigmoid=True), AsDiscrete(keys="pred", threshold=0.5)])
    post_label = Compose([AsDiscrete(threshold=0.5)])

    max_epochs = 50
    val_interval = 1 
    best_metric = -1
    best_metric_epoch = -1

    for epoch in range(max_epochs):
        print("-" * 10)
        print(f"Época {epoch + 1}/{max_epochs}")
        
        # --- FASE DE TREINO ---
        model.train()
        epoch_loss = 0
        step = 0
        
        for batch_data in train_loader:
            step += 1
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # Calcula a Loss combinada (Dice + Cross Entropy)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        epoch_loss /= step
        print(f"Loss média de Treino: {epoch_loss:.4f}")

        # --- FASE DE VALIDAÇÃO ---
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                for val_data in val_loader:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    
                    # Predição do modelo
                    val_outputs = model(val_inputs)
                    
                    # Aplica o pós-processamento (limiar de 0.5)
                    val_outputs = [post_pred(i) for i in val_outputs]
                    val_labels = [post_label(i) for i in val_labels]
                    
                    # Computa a métrica Dice para o batch atual
                    dice_metric(y_pred=val_outputs, y=val_labels)
                
                # Agrega o resultado de toda a validação
                metric = dice_metric.aggregate().item()
                dice_metric.reset() # Reseta para a próxima validação
                
                print(f"Métrica Dice na Validação: {metric:.4f}")
                
                # Salva o melhor modelo baseado no Score Dice
                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    torch.save(model.state_dict(), f"results/models/{model.__class__.__name__.lower()}/{dataset_name}_{model.__class__.__name__}.pth")
                    print("Encontrado um novo melhor modelo! Salvo com sucesso.")
    print(f"\nTreino concluído! Melhor Dice: {best_metric:.4f} na época {best_metric_epoch}")
                    




Iniciando treinamento para o dataset: MoNuSeg
----------
Época 1/50
Loss média de Treino: 1.2518
Métrica Dice na Validação: 0.2932
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 2/50
Loss média de Treino: 1.0762
Métrica Dice na Validação: 0.6525
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 3/50
Loss média de Treino: 0.9773
Métrica Dice na Validação: 0.7276
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 4/50
Loss média de Treino: 0.9323
Métrica Dice na Validação: 0.7499
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 5/50
Loss média de Treino: 0.9076
Métrica Dice na Validação: 0.7583
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 6/50
Loss média de Treino: 0.8923
Métrica Dice na Validação: 0.7626
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 7/50
Loss média de Treino: 0.8796
Métrica Dice na Validação: 0.7661
Encontrado um novo melhor modelo! Salvo 

# Treinando UNETR

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for dataset_name, dataset_splits in datasets.items():
    print(f"\nIniciando treinamento para o dataset: {dataset_name}")
    
    train_loader = DataLoader(dataset_splits['train'], batch_size=16, shuffle=True)
    val_loader = DataLoader(dataset_splits['val'], batch_size=16, shuffle=False)

    unetr = UNETR( # unetr do monai
        in_channels=3,
        out_channels=1,
        img_size=(256, 256),
        spatial_dims=2
    ).to(device)


    model = unetr

    loss_function = DiceCELoss(sigmoid=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    dice_metric = DiceMetric(include_background=True, reduction="mean")

    post_pred = Compose([Activations(keys="pred", sigmoid=True), AsDiscrete(keys="pred", threshold=0.5)])
    post_label = Compose([AsDiscrete(threshold=0.5)])

    max_epochs = 50
    val_interval = 1 
    best_metric = -1
    best_metric_epoch = -1

    for epoch in range(max_epochs):
        print("-" * 10)
        print(f"Época {epoch + 1}/{max_epochs}")
        
        # --- FASE DE TREINO ---
        model.train()
        epoch_loss = 0
        step = 0
        
        for batch_data in train_loader:
            step += 1
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # Calcula a Loss combinada (Dice + Cross Entropy)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        epoch_loss /= step
        print(f"Loss média de Treino: {epoch_loss:.4f}")

        # --- FASE DE VALIDAÇÃO ---
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                for val_data in val_loader:
                    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                    
                    # Predição do modelo
                    val_outputs = model(val_inputs)
                    
                    # Aplica o pós-processamento (limiar de 0.5)
                    val_outputs = [post_pred(i) for i in val_outputs]
                    val_labels = [post_label(i) for i in val_labels]
                    
                    # Computa a métrica Dice para o batch atual
                    dice_metric(y_pred=val_outputs, y=val_labels)
                
                # Agrega o resultado de toda a validação
                metric = dice_metric.aggregate().item()
                dice_metric.reset() # Reseta para a próxima validação
                
                print(f"Métrica Dice na Validação: {metric:.4f}")
                
                # Salva o melhor modelo baseado no Score Dice
                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    torch.save(model.state_dict(), f"results/models/{model.__class__.__name__.lower()}/{dataset_name}_{model.__class__.__name__}.pth")
                    print("Encontrado um novo melhor modelo! Salvo com sucesso.")
    print(f"\nTreino concluído! Melhor Dice: {best_metric:.4f} na época {best_metric_epoch}")
                    




Iniciando treinamento para o dataset: MoNuSeg
----------
Época 1/50
Loss média de Treino: 1.4113
Métrica Dice na Validação: 0.5013
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 2/50
Loss média de Treino: 1.2497
Métrica Dice na Validação: 0.5663
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 3/50
Loss média de Treino: 1.1507
Métrica Dice na Validação: 0.6139
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 4/50
Loss média de Treino: 1.0799
Métrica Dice na Validação: 0.6575
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 5/50
Loss média de Treino: 1.0205
Métrica Dice na Validação: 0.6957
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 6/50
Loss média de Treino: 0.9722
Métrica Dice na Validação: 0.7168
Encontrado um novo melhor modelo! Salvo com sucesso.
----------
Época 7/50
Loss média de Treino: 0.9417
Métrica Dice na Validação: 0.7356
Encontrado um novo melhor modelo! Salvo 